In [0]:
from pyspark.sql import functions as F

bronze_table = "main.`nyc-taxi`.bronze_taxi_trips"

bronze_df = spark.table(bronze_table)

print("=== BRONZE SCHEMA ===")
bronze_df.printSchema()

print("=== TOTAL ROWS ===")
print(bronze_df.count())

print("=== TOTAL COLUMNS ===")
print(len(bronze_df.columns))

print("=== COLUMN NAMES ===")
print(bronze_df.columns)

In [0]:
from pyspark.sql import functions as F

null_counts = bronze_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in bronze_df.columns
])

null_counts.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

print("=== DISTINCT EVENT IDs ===")

bronze_df.select(
    F.countDistinct("event_id").alias("distinct_event_ids")
).show()

print("=== DUPLICATED EVENT IDs ===")

bronze_df.groupBy("event_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(10, truncate=False)

In [0]:
duplicate_ids = [
    "4c4535874d9c7ba256b1e15efa097eed917017e4de5ea9ba82a2db0c9e9236ae",
    "d038b7ab63af2be463bf20a6c0887bfcd1a537513b7f7a22eec16f6462aa6d6b",
    "0e8c54bb748d107513d8d64da6dac585851f75091f7491f80e9ddd1b8d186634"
]

bronze_df.filter(
    F.col("event_id").isin(duplicate_ids)
).orderBy(
    "event_id",
    "source_file"
).show(10, truncate=False)

In [0]:
business_columns = [
    c for c in bronze_df.columns
    if c not in [
        "source_file",
        "ingestion_run_id",
        "ingested_at",
        "schema_version",
        "event_id"
    ]
]

print("Business columns:", len(business_columns))

duplicate_groups = (
    bronze_df
    .groupBy(*business_columns)
    .count()
    .filter(F.col("count") > 1)
)

print("=== NUMBER OF DUPLICATE GROUPS ===")
print(duplicate_groups.count())

print("=== TOTAL ROWS IN DUPLICATE GROUPS ===")
duplicate_groups.select(
    F.sum("count").alias("rows_in_duplicate_groups"),
    F.sum(F.col("count") - 1).alias("excess_rows")
).show()

In [0]:
financial_columns = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount"
]

bronze_df.select([
    F.min(c).alias(f"{c}_min")
    for c in financial_columns
] + [
    F.max(c).alias(f"{c}_max")
    for c in financial_columns
]).show(truncate=False)

In [0]:
components = (
    F.col("fare_amount")
    + F.col("extra")
    + F.col("mta_tax")
    + F.col("tip_amount")
    + F.col("tolls_amount")
    + F.col("improvement_surcharge")
)

financial_check = bronze_df.withColumn(
    "calculated_total",
    F.round(components, 2)
).withColumn(
    "total_difference",
    F.round(F.col("total_amount") - F.col("calculated_total"), 2)
)

print("=== FINANCIAL CONSISTENCY ===")

financial_check.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.abs(F.col("total_difference")) > 0.02, 1)
         .otherwise(0)
    ).alias("rows_with_difference"),
    F.max(F.abs(F.col("total_difference"))).alias("max_absolute_difference")
).show()

print("=== LARGEST DIFFERENCES ===")

financial_check.select(
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "calculated_total",
    "total_difference"
).orderBy(
    F.desc(F.abs(F.col("total_difference")))
).show(10, truncate=False)

In [0]:
financial_check.filter(
    F.abs(F.col("total_difference")) > 0.02
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "VendorID",
    "RatecodeID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "calculated_total",
    "total_difference"
).orderBy(
    F.desc(F.abs(F.col("total_difference")))
).show(30, truncate=False)

In [0]:
financial_check.filter(
    F.abs(F.col("total_difference")) > 0.02
).groupBy(
    "payment_type",
    "fare_amount",
    "total_amount"
).agg(
    F.count("*").alias("row_count"),
    F.max(
        F.abs(F.col("total_difference"))
    ).alias("max_absolute_difference")
).orderBy(
    F.desc("row_count")
).show(20, truncate=False)

In [0]:
financial_check.filter(
    (F.abs(F.col("total_difference")) > 0.02)
    & ~(
        (F.col("payment_type") == 2)
        & (F.col("fare_amount") == 0)
        & (F.col("total_amount") == 0)
    )
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "VendorID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "calculated_total",
    "total_difference"
).orderBy(
    F.desc(F.abs(F.col("total_difference")))
).show(30, truncate=False)

In [0]:
duration_df = bronze_df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

duration_df.select(
    F.min("trip_duration_minutes").alias("min_duration_minutes"),
    F.expr(
        "percentile_approx(trip_duration_minutes, 0.50)"
    ).alias("median_duration_minutes"),
    F.expr(
        "percentile_approx(trip_duration_minutes, 0.95)"
    ).alias("p95_duration_minutes"),
    F.max("trip_duration_minutes").alias("max_duration_minutes"),
    F.sum(
        F.when(F.col("trip_duration_minutes") <= 0, 1)
         .otherwise(0)
    ).alias("non_positive_duration_rows")
).show(truncate=False)

In [0]:
duration_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "passenger_count"
).orderBy(
    F.col("trip_duration_minutes").asc()
).show(10, truncate=False)

In [0]:
duration_df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "passenger_count"
).orderBy(
    F.col("trip_duration_minutes").desc()
).show(10, truncate=False)

In [0]:
duration_df.filter(
    F.col("trip_duration_minutes") > 0
).select(
    F.expr(
        "percentile_approx(trip_duration_minutes, 0.99)"
    ).alias("p99_duration_minutes"),
    F.expr(
        "percentile_approx(trip_duration_minutes, 0.999)"
    ).alias("p99_9_duration_minutes"),
    F.sum(
        F.when(F.col("trip_duration_minutes") > 60, 1)
         .otherwise(0)
    ).alias("trips_over_60_min"),
    F.sum(
        F.when(F.col("trip_duration_minutes") > 120, 1)
         .otherwise(0)
    ).alias("trips_over_120_min"),
    F.sum(
        F.when(F.col("trip_duration_minutes") > 1440, 1)
         .otherwise(0)
    ).alias("trips_over_24_hours")
).show(truncate=False)

In [0]:
silver_df = bronze_df.select(
    F.col("VendorID").alias("vendor_id"),
    F.col("tpep_pickup_datetime").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
    F.col("passenger_count").alias("passenger_count"),
    F.col("trip_distance").alias("trip_distance"),
    F.col("RatecodeID").alias("ratecode_id"),
    F.col("store_and_fwd_flag").alias("store_and_fwd_flag"),
    F.col("PULocationID").alias("pu_location_id"),
    F.col("DOLocationID").alias("do_location_id"),
    F.col("payment_type").alias("payment_type"),
    F.col("fare_amount").alias("fare_amount"),
    F.col("extra").alias("extra"),
    F.col("mta_tax").alias("mta_tax"),
    F.col("tip_amount").alias("tip_amount"),
    F.col("tolls_amount").alias("tolls_amount"),
    F.col("improvement_surcharge").alias("improvement_surcharge"),
    F.col("total_amount").alias("total_amount"),
    F.col("congestion_surcharge").alias("congestion_surcharge"),
    F.col("airport_fee").alias("airport_fee"),
    F.col("source_file"),
    F.col("ingestion_run_id"),
    F.col("ingested_at"),
    F.col("schema_version"),
    F.col("event_id")
).withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("dropoff_datetime")
        - F.unix_timestamp("pickup_datetime")
    ) / 60
).withColumn(
    "pickup_date",
    F.to_date("pickup_datetime")
).withColumn(
    "pickup_hour",
    F.hour("pickup_datetime")
).withColumn(
    "pickup_day_of_week",
    F.dayofweek("pickup_datetime")
)

In [0]:
silver_df.select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration_minutes",
    "pickup_date",
    "pickup_hour",
    "pickup_day_of_week"
).show(10, truncate=False)

print("Rows:", silver_df.count())
print("Columns:", len(silver_df.columns))

In [0]:
from pyspark.sql import functions as F

missing_timestamps = (
    F.col("pickup_datetime").isNull()
    | F.col("dropoff_datetime").isNull()
)

invalid_duration = (
    F.col("trip_duration_minutes") <= 0
)

invalid_passengers = (
    F.col("passenger_count").isNull()
    | (F.col("passenger_count") <= 0)
)

invalid_distance = (
    F.col("trip_distance").isNull()
    | (F.col("trip_distance") < 0)
)

silver_checked_df = silver_df.withColumn(
    "quarantine_reason",
    F.concat_ws(
        "; ",
        F.when(missing_timestamps, F.lit("missing_timestamp")),
        F.when(invalid_duration, F.lit("invalid_duration")),
        F.when(invalid_passengers, F.lit("invalid_passenger_count")),
        F.when(invalid_distance, F.lit("invalid_trip_distance"))
    )
)

quarantine_df = silver_checked_df.filter(
    F.col("quarantine_reason") != ""
)

silver_valid_df = silver_checked_df.filter(
    F.col("quarantine_reason") == ""
).drop("quarantine_reason")

print("Total Bronze-derived rows:", silver_checked_df.count())
print("Valid Silver rows:", silver_valid_df.count())
print("Quarantine rows:", quarantine_df.count())

In [0]:
quarantine_df.groupBy(
    "quarantine_reason"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

In [0]:
silver_table = "main.`nyc-taxi`.silver_taxi_trips"
quarantine_table = "main.`nyc-taxi`.quarantine_taxi_trips"

silver_valid_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

quarantine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(quarantine_table)

print("Silver and Quarantine tables saved successfully.")

In [0]:
silver_table = "main.`nyc-taxi`.silver_taxi_trips"
quarantine_table = "main.`nyc-taxi`.quarantine_taxi_trips"

print("Silver rows:", spark.table(silver_table).count())
print("Quarantine rows:", spark.table(quarantine_table).count())
print("Bronze rows:", spark.table(bronze_table).count())

In [0]:
silver_saved_df = spark.table(silver_table)

silver_saved_df.select(
    F.sum(
        F.when(
            F.col("trip_duration_minutes") <= 0, 1
        ).otherwise(0)
    ).alias("invalid_durations"),

    F.sum(
        F.when(
            F.col("passenger_count").isNull()
            | (F.col("passenger_count") <= 0), 1
        ).otherwise(0)
    ).alias("invalid_passenger_counts"),

    F.sum(
        F.when(
            F.col("pickup_datetime").isNull()
            | F.col("dropoff_datetime").isNull(), 1
        ).otherwise(0)
    ).alias("missing_timestamps"),

    F.sum(
        F.when(
            F.col("pickup_date") != F.to_date("pickup_datetime"), 1
        ).otherwise(0)
    ).alias("incorrect_pickup_dates")
).show()

In [0]:
business_columns = [
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "ratecode_id",
    "store_and_fwd_flag",
    "pu_location_id",
    "do_location_id",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee"
]

duplicate_groups = (
    silver_saved_df
    .groupBy(*business_columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_groups.select(
    F.count("*").alias("duplicate_groups"),
    F.sum("count").alias("rows_in_duplicate_groups"),
    F.sum(F.col("count") - 1).alias("excess_rows")
).show()

In [0]:
spark.table(silver_table).printSchema()

In [0]:
from pyspark.sql import functions as F

silver_saved_df = spark.table("main.`nyc-taxi`.silver_taxi_trips")
quarantine_saved_df = spark.table("main.`nyc-taxi`.quarantine_taxi_trips")
bronze_saved_df = spark.table("main.`nyc-taxi`.bronze_taxi_trips")

# Test 1: conservation du nombre total de lignes
assert (
    silver_saved_df.count() + quarantine_saved_df.count()
    == bronze_saved_df.count()
), "FAIL: Silver + Quarantine != Bronze"

# Test 2: aucune durée non positive dans Silver
assert silver_saved_df.filter(
    F.col("trip_duration_minutes") <= 0
).count() == 0, "FAIL: invalid durations in Silver"

# Test 3: aucun nombre de passagers nul ou négatif
assert silver_saved_df.filter(
    F.col("passenger_count").isNull()
    | (F.col("passenger_count") <= 0)
).count() == 0, "FAIL: invalid passenger counts in Silver"

# Test 4: aucune ligne sans motif dans la quarantaine
assert quarantine_saved_df.filter(
    F.col("quarantine_reason").isNull()
    | (F.trim(F.col("quarantine_reason")) == "")
).count() == 0, "FAIL: quarantine rows without a reason"

print("PASS: all 4 Silver quality tests succeeded.")